# Assignment 06 - Weather Analysis

This notebook analyzes daily weather trends for Winter Wheat fields in Lea, Roosevelt, and Curry counties using available 2005-2020 NASA POWER data.

Focus window for this assignment: **September through June** seasons between **2015-09-01 and 2020-06-30**.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
DASHBOARD_ASSETS_DIR = OUTPUT_DIR / "dashboard_assets"
DASHBOARD_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

weather_path = DATA_DIR / "weather" / "nm_weather_2005_2020.csv"
crop_path = DATA_DIR / "crops" / "nm_cdl_2008_2020.csv"
boundaries_path = DATA_DIR / "boundaries" / "nm_top_200_fields.geojson"

print(f"Project root: {PROJECT_ROOT}")
print(f"Weather path: {weather_path}")
print(f"Dashboard assets: {DASHBOARD_ASSETS_DIR}")

In [ ]:
weather_df = pd.read_csv(weather_path, parse_dates=["date"])
crops_df = pd.read_csv(crop_path)

with open(boundaries_path, encoding="utf-8") as f:
    boundaries = json.load(f)

county_map = {
    feature["properties"]["field_id"]: feature["properties"]["county"]
    for feature in boundaries.get("features", [])
}

print("Datetime dtype:", weather_df["date"].dtype)
print("Weather date range:", weather_df["date"].min(), "to", weather_df["date"].max())
print("Weather records:", len(weather_df))

In [ ]:
winter_wheat = crops_df[crops_df["crop_name"].eq("Winter Wheat")].copy()
winter_wheat["county"] = winter_wheat["field_id"].map(county_map)

field_rank = (
    winter_wheat.groupby(["county", "field_id"], as_index=False)
    .size()
    .rename(columns={"size": "years_as_winter_wheat"})
    .sort_values(["county", "years_as_winter_wheat", "field_id"], ascending=[True, False, True])
)

target_counties = ["Lea", "Roosevelt", "Curry"]
selected_fields_df = (
    field_rank[field_rank["county"].isin(target_counties)]
    .groupby("county", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)
selected_fields = selected_fields_df["field_id"].tolist()

print("Selected Winter Wheat fields (up to 3 per county):")
display(selected_fields_df)

In [ ]:
analysis_start = pd.Timestamp("2015-09-01")
analysis_end = pd.Timestamp("2020-06-30")
season_months = {9, 10, 11, 12, 1, 2, 3, 4, 5, 6}

analysis_df = weather_df[weather_df["field_id"].isin(selected_fields)].copy()
analysis_df["county"] = analysis_df["field_id"].map(county_map)
analysis_df = analysis_df[
    (analysis_df["date"] >= analysis_start)
    & (analysis_df["date"] <= analysis_end)
    & (analysis_df["date"].dt.month.isin(season_months))
].copy()

analysis_df["high_f"] = (analysis_df["T2M_MAX"] * 9 / 5) + 32
analysis_df["precip_mm"] = analysis_df["PRECTOTCORR"].fillna(0.0)
analysis_df = analysis_df.sort_values(["county", "field_id", "date"]).reset_index(drop=True)

analysis_df["high_f_roll15"] = analysis_df.groupby("field_id")["high_f"].transform(
    lambda s: s.rolling(window=15, min_periods=1).mean()
)
analysis_df["precip_roll15"] = analysis_df.groupby("field_id")["precip_mm"].transform(
    lambda s: s.rolling(window=15, min_periods=1).mean()
)

analysis_df["is_abnormal_heat"] = analysis_df["high_f"] > 105
abnormal_days = analysis_df[analysis_df["is_abnormal_heat"]].copy()

print("Filtered records:", len(analysis_df))
print("Date span in filtered data:", analysis_df["date"].min(), "to", analysis_df["date"].max())
print("Abnormal heat days (>105F):", len(abnormal_days))

In [ ]:
abnormal_table = abnormal_days[["date", "county", "field_id", "high_f", "precip_mm"]].copy()
abnormal_table["high_f"] = abnormal_table["high_f"].round(1)
abnormal_table["precip_mm"] = abnormal_table["precip_mm"].round(2)

abnormal_csv_path = DASHBOARD_ASSETS_DIR / "weather_abnormal_days.csv"
abnormal_table.to_csv(abnormal_csv_path, index=False)

display(abnormal_table.head(50))
print(f"Saved abnormal-days table: {abnormal_csv_path}")

In [ ]:
county_order = ["Lea", "Roosevelt", "Curry"]
palette = sns.color_palette("tab10", n_colors=max(3, analysis_df["field_id"].nunique()))

fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True)

for i, county in enumerate(county_order):
    ax = axes[i]
    county_df = analysis_df[analysis_df["county"] == county]

    for j, (field_id, field_df) in enumerate(county_df.groupby("field_id")):
        color = palette[j % len(palette)]
        ax.plot(
            field_df["date"],
            field_df["high_f"],
            color=color,
            linewidth=1.4,
            alpha=0.9,
            label=f"{field_id} - daily high (F)",
        )

    precip_daily = county_df.groupby("date", as_index=False)["precip_mm"].mean()
    ax2 = ax.twinx()
    ax2.bar(
        precip_daily["date"],
        precip_daily["precip_mm"],
        width=1.0,
        alpha=0.18,
        color="#1f77b4",
        label="Mean precip (mm)",
    )
    ax2.set_ylabel("Precipitation (mm)", color="#1f77b4")

    ax.set_title(
        f"{county} County - Daily High Temperature and Precipitation",
        fontsize=12,
        fontweight="bold",
    )
    ax.set_ylabel("High temperature (F)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=8, frameon=True)

axes[-1].set_xlabel("Date")
fig.suptitle(
    "Raw Daily Weather Trends (Sep-Jun Seasons, 2015-2020)", fontsize=16, fontweight="bold"
)
fig.tight_layout(rect=[0, 0, 1, 0.97])

raw_out = DASHBOARD_ASSETS_DIR / "weather_trends_raw.png"
fig.savefig(raw_out, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved raw trends figure: {raw_out}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True)

for i, county in enumerate(county_order):
    ax = axes[i]
    county_df = analysis_df[analysis_df["county"] == county]

    for j, (field_id, field_df) in enumerate(county_df.groupby("field_id")):
        color = palette[j % len(palette)]
        ax.plot(
            field_df["date"],
            field_df["high_f_roll15"],
            color=color,
            linewidth=2.2,
            alpha=0.95,
            label=f"{field_id} - 15-day high avg (F)",
            zorder=3,
        )

    precip_roll = county_df.groupby("date", as_index=False)["precip_roll15"].mean()
    ax2 = ax.twinx()
    ax2.plot(
        precip_roll["date"],
        precip_roll["precip_roll15"],
        color="#1f77b4",
        linewidth=1.5,
        linestyle="--",
        alpha=0.8,
        label="15-day precip avg (mm)",
        zorder=1,
    )
    ax2.set_ylabel("15-day precipitation avg (mm)", color="#1f77b4")

    ax.set_zorder(2)
    ax2.set_zorder(1)
    ax.patch.set_alpha(0)

    abnormal_county = county_df[county_df["is_abnormal_heat"]]
    if not abnormal_county.empty:
        ax.scatter(
            abnormal_county["date"],
            abnormal_county["high_f_roll15"],
            s=70,
            color="#d62728",
            edgecolor="black",
            linewidth=0.8,
            label="Abnormal day (>105F)",
            zorder=10,
        )

    ax.axhline(105, color="#d62728", linestyle=":", linewidth=1.2, alpha=0.8, zorder=2)
    ax.set_title(
        f"{county} County - Smoothed Temperature and Precipitation", fontsize=12, fontweight="bold"
    )
    ax.set_ylabel("15-day high temp avg (F)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=8, frameon=True)

axes[-1].set_xlabel("Date")
fig.suptitle(
    "Smoothed Weather Trends + Abnormal Heat Highlights (Sep-Jun Seasons, 2015-2020)",
    fontsize=16,
    fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 0.97])

smoothed_out = DASHBOARD_ASSETS_DIR / "weather_trends.png"
fig.savefig(smoothed_out, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved smoothed trends figure: {smoothed_out}")

## Interpretation

The selected Winter Wheat weather trends show strong seasonal temperature swings, with summer-edge periods driving most high-temperature stress events.

The 15-day smoothing makes county-level patterns easier to compare while preserving field-level differences. Days exceeding 105F are highlighted to support rapid stress-period inspection for management decisions.

Because available weather data ends in 2020 and Winter Wheat coverage in Lea is sparse, the analysis uses the best available field coverage per county.